# Powston Simulator — quickstart

Run a single simulation against a week of your own meter data and see the plot. This is the smaller, faster cousin of `run-site-sim.py` — one run with the variables your live rule already has, no tuning loop.

If this works end to end, the full tuner will too. Use this notebook to confirm your API key, inverter ID, and meter data are all wired correctly **before** kicking off the long differential-evolution run.

**Before running:** `POWSTON_API_KEY` must be set as a Codespaces secret on your fork. See the README.

In [ ]:
import json
import os
import re

from aemo_to_tariff import spot_to_tariff
from powston_simulator.utils import plot, run_scripted_simulation
from powston_simulator.sim_utils import (
    check_local_paths,
    get_active_user_code,
    get_inverter_params,
    get_meter_inverter_data,
    save_user_code,
    site_info,
)

assert os.environ.get("POWSTON_API_KEY"), "POWSTON_API_KEY is not set."
check_local_paths()
print("Setup OK.")

## Choose an inverter

Edit these values. `DAYS_AGO` is where the simulation window starts (days back from today); `DAYS` is the window length.

In [ ]:
INVERTER_ID = 3
DAYS_AGO = 7
DAYS = 7
BATTERY_LOSS = 40  # default round-trip loss percent if the API doesn't supply one

## Fetch the active rule, site, and meter data

In [ ]:
_, site_id, rule_name, inverter_code, last_training_impact = get_active_user_code(INVERTER_ID)
print(f"Rule:  {rule_name}")
print(f"Site:  {site_id}")

file_name = save_user_code(INVERTER_ID, inverter_code["user_code"])
print(f"User code saved to {file_name}")

meter_data_df, interval = get_meter_inverter_data(INVERTER_ID, days_ago=DAYS_AGO, days=DAYS)
print(
    f"Meter rows: {len(meter_data_df)} "
    f"({meter_data_df.index.min()} -> {meter_data_df.index.max()}, interval {interval} min)"
)

## Site and inverter parameters

In [ ]:
network, tariff, export_tariff, daily_fee, latitude, longitude, state, timezone_str = site_info(site_id)
(
    export_limit_max, installed_battery, installed_solar, inverter_power,
    timezone, latitude, longitude, battery_capacity, charge_rate,
    max_ppv_power, battery_loss,
) = get_inverter_params(INVERTER_ID)

if not battery_loss:
    battery_loss = BATTERY_LOSS

print(f"Network/tariff:    {network} / {tariff}")
print(f"Battery capacity:  {battery_capacity} Wh")
print(f"Charge rate:       {charge_rate} W")
print(f"Installed solar:   {installed_solar} W")
print(f"Round-trip loss:   {battery_loss}%")

## Run a single simulation with the live rule

In [ ]:
daily_fee = max(daily_fee, meter_data_df["billed_fixed_costs"].sum() / DAYS)
battery_charge = meter_data_df["start_battery_soc"].iloc[0] * battery_capacity / 100.0
grid_limit = charge_rate * 1.5

meter_data_df["forecast"] = meter_data_df["forecasts"]
meter_data_df["export_limit_max"] = export_limit_max
meter_data_df["feed_in_power_limitation"] = meter_data_df["inverter_params"].apply(
    lambda x: json.loads(x).get("feed_in_power_limitation", export_limit_max) if x else export_limit_max
)

with open(file_name) as f:
    script_content = f.read()

script_bill, ret_df = run_scripted_simulation(
    meter_data_df, script_content, file_name, interval=interval,
    battery_capacity=battery_capacity, tariff=tariff, export_tariff=export_tariff, network=network,
    charge_rate=charge_rate, max_ppv_power=max_ppv_power, daily_fee=daily_fee,
    spot_to_tariff=spot_to_tariff, state=state, battery_loss=battery_loss,
    latitude=latitude, longitude=longitude, timezone_str=timezone,
    battery_charge=battery_charge, grid_limit=grid_limit,
)

metered_bill = (meter_data_df["billed_costs"].sum() - meter_data_df["billed_earnings"].sum()) / 100
print(f"Simulated bill (current rule): ${script_bill / 100:,.2f}")
print(f"Metered bill (what you paid): ${metered_bill:,.2f}")

## Plot

In [ ]:
plot(ret_df)

## Tweak a variable and re-run

Substitute one variable and run again to see the immediate effect on the bill. For full optimisation across many variables, use `run-site-sim.py`.

In [ ]:
override = {"BATTERY_SOC_NEEDED": 50}

modified = script_content
for k, v in override.items():
    modified = re.sub(rf"(?m)^{k}\s*=.*$", f"{k} = {int(v)}", modified)

bill2, ret_df2 = run_scripted_simulation(
    meter_data_df, modified, file_name, interval=interval,
    battery_capacity=battery_capacity, tariff=tariff, export_tariff=export_tariff, network=network,
    charge_rate=charge_rate, max_ppv_power=max_ppv_power, daily_fee=daily_fee,
    spot_to_tariff=spot_to_tariff, state=state, battery_loss=battery_loss,
    latitude=latitude, longitude=longitude, timezone_str=timezone,
    battery_charge=battery_charge, grid_limit=grid_limit,
)

delta = (script_bill - bill2) / 100
print(f"With {override}: ${bill2 / 100:,.2f}  (delta vs baseline: ${delta:+,.2f})")
plot(ret_df2)